In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install func_timeout

In [ ]:
import json
import sqlite3
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch
import random
import time
from func_timeout import func_timeout, FunctionTimedOut
from collections import Counter

In [ ]:
MODEL_PATH = "path/to/your-model"
DEV_JSON_PATH = "path/to/dev/dev_20240627/dev.json"
DB_ROOT_PATH = "path/to/dev/dev_20240627/dev_databases/dev_databases"
print("Loading Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
# If set to None then run all 1534 samples
TEST_LIMIT = 200
# Helper Function：Extract Schema from database
def get_schema_prompt(db_path, db_name):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Get all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()

        schema_str = ""
        for table in tables:
            table_name = table[0]
            # Get the create table statement
            cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
            create_stmt = cursor.fetchone()[0]
            schema_str += create_stmt + ";\n"

        conn.close()
        return schema_str
    except Exception as e:
        return ""

# Run the SQL and get the result
def _run_query(db_path, sql):
      conn = sqlite3.connect(db_path)
      cursor = conn.cursor()
      cursor.execute(sql)
      result = cursor.fetchall()
      conn.close()
      # Use set to ignore order
      return set(result)

# Execute SQL with time limit
def execute_sql(db_path, sql):
    try:
        # 10 second time limit
        return func_timeout(10, _run_query, args=(db_path, sql))
    except FunctionTimedOut:
        return "Error: Timeout (SQL took too long)"
    except Exception as e:
        return f"Error: {e}"

# Main Eval loop
print(f"Starting Evaluation... (Limited to {TEST_LIMIT} samples)" if TEST_LIMIT else "Starting Full Evualation...")

with open(DEV_JSON_PATH, 'r') as f:
    dev_data = json.load(f)

if TEST_LIMIT:
    # Random Sampling
    random.seed(42)
    dev_data = random.sample(dev_data, TEST_LIMIT)

correct_count = 0
total_count = 0
results_log = []

for item in tqdm(dev_data):
    start_time = time.time()
    question = item['question']
    evidence = item['evidence']
    db_id = item['db_id']
    difficulty = item['difficulty']
    gold_sql = item['SQL'] # Correct answer

    # Build the database
    db_path = f"{DB_ROOT_PATH}/{db_id}/{db_id}.sqlite"

    # Get Schema
    schema = get_schema_prompt(db_path, db_id)

    # Prompt, same format as training
    prompt = f"""<|im_start|>system
You are a SQL expert. Generate a SQL query to answer the question based on the database schema provided.<|im_end|>
<|im_start|>user
{schema}

Hint: {evidence}

Question: {question}<|im_end|>
<|im_start|>assistant
"""

    # Prepare Inputs
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate Candidates (Self-Consistency)
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            num_return_sequences=8,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    except RuntimeError as e:
        print(f"OOM Error: {e}. Reducing batch size...")
        # Fallback logic could go here
        pred_sql = "SELECT * FROM Error"

    # Process & Vote
    candidates = []

    input_len = inputs.input_ids.shape[1]

    # multiple answers, needs to parse each
    for output_seq in outputs:
        generated_text = tokenizer.decode(output_seq[input_len:], skip_special_tokens=True)

        # Clean SQL
        cleaned_sql = generated_text.replace("```sql", "").replace("```", "").strip()
        # Remove any trailing junk
        cleaned_sql = cleaned_sql.split(";")[0].strip()

        if cleaned_sql:
            candidates.append(cleaned_sql)

    # Majority Voting
    if not candidates:
        pred_sql = "SELECT * FROM T1" # Ultimate fallback
    else:
        # Get vote counts
        vote_counts = Counter(candidates)
        # Most common SQL
        pred_sql = vote_counts.most_common(1)[0][0]
        # Get Vote Distribution
        counts = [count for _, count in vote_counts.most_common()]
        vote_distribution = str(counts)

    # Execution Accuracy Check
    pred_result = execute_sql(db_path, pred_sql)
    gold_result = execute_sql(db_path, gold_sql)

    is_correct = False
    # If both Error, count as False
    if isinstance(pred_result, str) or isinstance(gold_result, str):
        is_correct = False
    else:
        # Check if the result is the same
        is_correct = (pred_result == gold_result)

    if is_correct:
        correct_count += 1

    total_count += 1

    # Stop the timer
    elapsed_time = time.time() - start_time

    # Print Result Immediately
    icon = "✅" if is_correct else "❌"
    status_msg = "Correct" if is_correct else "Wrong"
    if isinstance(pred_result, str): status_msg = "Exec Error"
    current_accuracy = (correct_count / total_count) * 100
    print(f" {icon} [{status_msg}] Time: {elapsed_time:.2f}s | DB: {db_id} | Current Accuracy: {current_accuracy:.2f}% | Votes: {vote_distribution}")

    # Log
    results_log.append({
        "question": question,
        "evidence": evidence,
        "db": db_id,
        "pred_sql": pred_sql,
        "gold_sql": gold_sql,
        "is_correct": is_correct,
        "difficulty": difficulty,
        "vote_distribution": vote_distribution
    })

# Print Final Report
accuracy = (correct_count / total_count) * 100
print("\n" + "="*30)
print(f"BIRD-Dev Eval Result")
print("="*30)
print(f"Tested Sample Number: {total_count}")
print(f"Correct Count: {correct_count}")
print(f"Execution Accuracy: {accuracy:.2f}%")
print("="*30)

# Save result for error analysis
df_results = pd.DataFrame(results_log)
save_path = "bird_eval_results.csv"
df_results.to_csv(save_path, index=False)
print(f"Detailed result saved to: {save_path}")